In [1]:
import os
import cv2
import torch
import random
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.model_selection import train_test_split
from datetime import datetime
import json

CONFIG = {
    "model_name": "google/vit-base-patch16-224-in21k",
    "num_epochs": 10,
    "batch_size": 32,
    "learning_rate": 2e-5,
    "num_workers": 0,
    "fake_video_count": 1000,       # fake 영상 샘플링 수
    "fake_frames_per_video": 100,   # fake 영상당 프레임 수
    "real_frames_per_video": 1000,  # real 영상당 프레임 수
    "frame_interval": 5,            # 몇 프레임마다 추출할지
    "fake_video_path": r"C:\Users\user\Desktop\deepfake-detector\data\vidprom\pika_fake\pika_videos_all",
    "real_video_path": r"C:\Users\user\Desktop\deepfake-detector\data\vidprom\real_videos",
    "frame_save_path": r"C:\Users\user\Desktop\deepfake-detector\data\vidprom\frames",
    "model_save_path": r"C:\Users\user\Desktop\deepfake-detector\models\vidprom",
    "best_model_path": r"C:\Users\user\Desktop\deepfake-detector\models\vidprom\best_model.pth",
    "checkpoint_path": r"C:\Users\user\Desktop\deepfake-detector\models\vidprom\checkpoint.pth",
    "history_path":    r"C:\Users\user\Desktop\deepfake-detector\models\vidprom\history.json",
}

os.makedirs(CONFIG["frame_save_path"] + "/fake", exist_ok=True)
os.makedirs(CONFIG["frame_save_path"] + "/real", exist_ok=True)
os.makedirs(CONFIG["model_save_path"], exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
print("설정 완료!")

device: cuda
설정 완료!


In [2]:
import os
import cv2
import random
import json
from datetime import datetime

def extract_frames(video_path, save_dir, video_id, frames_per_video, frame_interval):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    saved = 0
    frame_idx = 0
    while saved < frames_per_video and frame_idx < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        save_path = os.path.join(save_dir, f"{video_id}_frame{saved:04d}.jpg")
        cv2.imwrite(save_path, frame)
        saved += 1
        frame_idx += frame_interval
    cap.release()
    return saved

# 진행 상황 저장/로드 경로
progress_path = r"C:\Users\user\Desktop\deepfake-detector\data\vidprom\extract_progress.json"

def load_progress():
    if os.path.exists(progress_path):
        with open(progress_path, "r") as f:
            return json.load(f)
    return {"fake_done": [], "real_done": []}

def save_progress(progress):
    with open(progress_path, "w") as f:
        json.dump(progress, f)

progress = load_progress()
print(f"이전 진행 상황 로드: fake {len(progress['fake_done'])}개, real {len(progress['real_done'])}개 완료")

# fake 영상 랜덤 샘플링
all_fake_videos = [f for f in os.listdir(CONFIG["fake_video_path"]) if f.endswith(".mp4")]
random.seed(42)
sampled_fake_videos = random.sample(all_fake_videos, min(CONFIG["fake_video_count"], len(all_fake_videos)))
print(f"fake 전체: {len(all_fake_videos)}개 → 샘플링: {len(sampled_fake_videos)}개")

total_fake_frames = 0
start_time = datetime.now()

for i, fname in enumerate(sampled_fake_videos):
    if fname in progress["fake_done"]:
        continue  # 이미 완료된 영상 스킵

    video_id = fname.replace(".mp4", "")
    video_path = os.path.join(CONFIG["fake_video_path"], fname)
    saved = extract_frames(video_path, CONFIG["frame_save_path"] + "/fake", video_id,
                           CONFIG["fake_frames_per_video"], CONFIG["frame_interval"])
    total_fake_frames += saved
    progress["fake_done"].append(fname)

    if (i+1) % 100 == 0:
        elapsed = (datetime.now() - start_time).seconds // 60
        print(f"[fake] {i+1}/{len(sampled_fake_videos)} 완료 | 누적 프레임: {total_fake_frames}장 | 경과: {elapsed}분")
        save_progress(progress)

save_progress(progress)
print(f"fake 프레임 추출 완료! 총 {total_fake_frames}장")

# real 프레임 추출
real_videos = [f for f in os.listdir(CONFIG["real_video_path"]) if f.endswith(".mp4")]
print(f"\nreal 영상 수: {len(real_videos)}개")

total_real_frames = 0
start_time = datetime.now()

for i, fname in enumerate(real_videos):
    if fname in progress["real_done"]:
        continue  # 이미 완료된 영상 스킵

    video_id = fname.replace(".mp4", "")
    video_path = os.path.join(CONFIG["real_video_path"], fname)
    saved = extract_frames(video_path, CONFIG["frame_save_path"] + "/real", video_id,
                           CONFIG["real_frames_per_video"], CONFIG["frame_interval"])
    total_real_frames += saved
    progress["real_done"].append(fname)

    # 영상마다 출력 + 1000프레임 단위 누적 출력
    elapsed = (datetime.now() - start_time).seconds // 60
    print(f"[real] {i+1}/{len(real_videos)} | {fname} | {saved}프레임 추출 | 누적: {total_real_frames}장 | 경과: {elapsed}분")
    save_progress(progress)

save_progress(progress)
print(f"\nreal 프레임 추출 완료! 총 {total_real_frames}장")

fake_frames = os.listdir(CONFIG["frame_save_path"] + "/fake")
real_frames = os.listdir(CONFIG["frame_save_path"] + "/real")
print(f"\nfake 프레임 수: {len(fake_frames)}")
print(f"real 프레임 수: {len(real_frames)}")

이전 진행 상황 로드: fake 1000개, real 45개 완료
fake 전체: 57663개 → 샘플링: 1000개
fake 프레임 추출 완료! 총 0장

real 영상 수: 45개

real 프레임 추출 완료! 총 0장

fake 프레임 수: 12521
real 프레임 수: 44996


In [8]:
processor = ViTImageProcessor.from_pretrained(CONFIG["model_name"])
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

class VideoFrameDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# 파일 목록 구성
fake_dir = CONFIG["frame_save_path"] + "/fake"
real_dir = CONFIG["frame_save_path"] + "/real"

fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(".jpg")]
real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(".jpg")]

# real/fake 수 맞추기 (언더샘플링)
min_count = min(len(fake_files), len(real_files))
import random
random.seed(42)
fake_files = random.sample(fake_files, min_count)
real_files = random.sample(real_files, min_count)

all_files = fake_files + real_files
all_labels = [1] * len(fake_files) + [0] * len(real_files)

train_files, val_files, train_labels, val_labels = train_test_split(
    all_files, all_labels, test_size=0.1, random_state=42, stratify=all_labels
)

train_dataset = VideoFrameDataset(train_files, train_labels, transform=train_transform)
val_dataset   = VideoFrameDataset(val_files,   val_labels,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"])
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

print(f"train: {len(train_dataset)}장 / val: {len(val_dataset)}장")
print(f"train batches: {len(train_loader)} / val batches: {len(val_loader)}")

train: 22537장 / val: 2505장
train batches: 705 / val batches: 79


In [9]:
model = ViTForImageClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2,
    ignore_mismatched_sizes=True
)
model = model.to(device)
print("모델 로드 완료!")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델 로드 완료!


In [10]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 100 == 0:
            print(f"  batch {batch_idx}/{len(loader)} | loss: {loss.item():.4f}")

    return total_loss / len(loader), correct / total

In [11]:
from sklearn.metrics import f1_score

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, labels)

            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds)
    return total_loss / len(loader), correct / total, f1

In [12]:
def save_checkpoint(epoch, model, optimizer, history, best_val_acc):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_acc": best_val_acc,
    }, CONFIG["checkpoint_path"])
    with open(CONFIG["history_path"], "w") as f:
        json.dump(history, f)
    print(f"체크포인트 저장: epoch {epoch}")

def load_checkpoint(model, optimizer):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    if not os.path.exists(CONFIG["checkpoint_path"]):
        print("체크포인트 없음 - 처음부터 학습 시작")
        return 0, history, 0.0
    ckpt = torch.load(CONFIG["checkpoint_path"], map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if os.path.exists(CONFIG["history_path"]):
        with open(CONFIG["history_path"], "r") as f:
            history = json.load(f)
    print(f"체크포인트 로드: epoch {ckpt['epoch']+1}부터 재개")
    return ckpt["epoch"] + 1, history, ckpt["best_val_acc"]

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])
criterion = nn.CrossEntropyLoss()
start_epoch, history, best_val_acc = load_checkpoint(model, optimizer)
training_start_time = datetime.now()
print(f"\n학습 시작: epoch {start_epoch} ~ {CONFIG['num_epochs']-1}")
print(f"학습 시작 시각: {training_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*50)

for epoch in range(start_epoch, CONFIG["num_epochs"]):
    print(f"\n[Epoch {epoch+1}/{CONFIG['num_epochs']}]")
    start_time = datetime.now()

    # 학습
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # 10배치마다 진행률 출력
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(train_loader):
            progress = (batch_idx + 1) / len(train_loader) * 100
            batch_elapsed = (datetime.now() - start_time).seconds
            print(f"  [train] {batch_idx+1}/{len(train_loader)} ({progress:.1f}%) | loss: {loss.item():.4f} | 경과: {batch_elapsed}초")

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    # 검증
    val_loss, val_acc, val_f1 = validate(model, val_loader, criterion, device)

    elapsed = (datetime.now() - start_time).seconds // 60
    total_elapsed = (datetime.now() - training_start_time).seconds // 60

    print(f"  train loss: {train_loss:.4f} | train acc: {train_acc*100:.2f}%")
    print(f"  val loss:   {val_loss:.4f} | val acc:   {val_acc*100:.2f}% | val F1: {val_f1:.4f}")
    print(f"  에폭 소요시간: {elapsed}분 | 전체 경과: {total_elapsed}분")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CONFIG["best_model_path"])
        print(f"  ✅ 최고 모델 저장! val acc: {val_acc*100:.2f}%")

    save_checkpoint(epoch, model, optimizer, history, best_val_acc)

print("\n학습 완료!")
print(f"최고 val acc: {best_val_acc*100:.2f}%")
print(f"총 학습 시간: {(datetime.now() - training_start_time).seconds // 60}분")

체크포인트 로드: epoch 10부터 재개

학습 시작: epoch 10 ~ 9
학습 시작 시각: 2026-03-30 16:09:37

학습 완료!
최고 val acc: 100.00%
총 학습 시간: 0분


In [15]:
from transformers import ViTImageProcessor
processor = ViTImageProcessor.from_pretrained(CONFIG["model_name"])
import torch
from PIL import Image
from torchvision import transforms
from transformers import ViTForImageClassification, ViTImageProcessor

def predict_image(image_path, model, processor, device):
    model.eval()
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    ])
    
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(pixel_values=img_tensor)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = probs.argmax(dim=1).item()
        confidence = probs[0][pred].item() * 100
    
    label = "FAKE" if pred == 1 else "REAL"
    print(f"판별 결과: {label} ({confidence:.2f}%)")
    return label, confidence

# 테스트할 이미지 경로 입력
image_path = r"테스트할 이미지 경로"
predict_image(image_path, model, processor, device)

FileNotFoundError: [Errno 2] No such file or directory: '테스트할 이미지 경로'

In [14]:
from transformers import ViTImageProcessor
processor = ViTImageProcessor.from_pretrained(CONFIG["model_name"])
def predict_video(video_path, model, processor, device, sample_frames=10):
    model.eval()
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    ])
    
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = total_frames // sample_frames
    
    fake_scores = []
    
    for i in range(sample_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
        ret, frame = cap.read()
        if not ret:
            break
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        img_tensor = transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = model(pixel_values=img_tensor)
            probs = torch.softmax(outputs.logits, dim=1)
            fake_scores.append(probs[0][1].item())
    
    cap.release()
    
    avg_fake_score = sum(fake_scores) / len(fake_scores) * 100
    label = "FAKE" if avg_fake_score > 50 else "REAL"
    print(f"판별 결과: {label} ({avg_fake_score:.2f}%)")
    return label, avg_fake_score

# 테스트할 영상 경로 입력
video_path = r"테스트할 영상 경로"
predict_video(video_path, model, processor, device)

ZeroDivisionError: division by zero